# Gaussian mixture models

k-means says each point belongs to exactly one cluster. A **Gaussian mixture model** (GMM) says each point was drawn from one of several bell-shaped distributions and reports the probability of each. That gives soft assignments, clusters of different sizes and shapes, and a proper likelihood, which lets us choose the number of clusters with a principled score. It is also the cleanest example of the expectation-maximisation (EM) algorithm, which shows up again in later sections.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal, norm
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture

from common.datasets import make_moons_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Heights of adults are not one bell curve. Mix men and women and you get one hump with a shoulder, which is two bell curves added together. If you knew who was who you could fit each curve separately. If you knew each curve you could work out, for any height, how likely it came from each group. You know neither, so you guess one and use it to improve the other. That loop is EM.

A point at the border of two clusters should not be forced into one. A GMM reports something like "62% cluster A, 38% cluster B". That number is the **responsibility** cluster A takes for the point.

## 2. Maths and cost function

A GMM with $K$ components says the density of a point is a weighted sum of Gaussians:

$$p(\mathbf x)=\sum_{k=1}^{K}\pi_k\,\mathcal N(\mathbf x\mid\boldsymbol\mu_k,\Sigma_k),\qquad \pi_k\ge 0,\ \sum_k\pi_k=1$$

Each component has a weight $\pi_k$ (how common it is), a mean $\boldsymbol\mu_k$ and a covariance $\Sigma_k$ (its size and tilt). This is the model for which we want the **maximum likelihood** parameters, the same idea as in `02_probability/03_probability_in_ml.ipynb`. The cost to minimise is the negative log-likelihood:

$$-\ell=-\sum_{i=1}^{n}\log\sum_{k=1}^{K}\pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)$$

For a single Gaussian, setting the gradient to zero gives the sample mean and covariance in closed form. Here a sum sits inside the log, so there is no closed form. EM sidesteps it by alternating:

**E step.** With the current parameters, compute each component's responsibility for each point, by Bayes' rule (see `02_probability/02_bayes.ipynb`):

$$r_{ik}=\frac{\pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)}{\sum_j\pi_j\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_j,\Sigma_j)}$$

**M step.** Re-fit each Gaussian by maximum likelihood, but with every point counted in proportion to its responsibility. With $N_k=\sum_i r_{ik}$:

$$\pi_k=\frac{N_k}{n},\qquad \boldsymbol\mu_k=\frac1{N_k}\sum_i r_{ik}\mathbf x_i,\qquad \Sigma_k=\frac1{N_k}\sum_i r_{ik}(\mathbf x_i-\boldsymbol\mu_k)(\mathbf x_i-\boldsymbol\mu_k)^\top$$

Each full loop never lowers the likelihood, so it climbs to a local maximum. As with k-means, where it ends depends on where it starts.

**The link to k-means.** Fix every covariance to $\sigma^2 I$ with the same $\sigma$ and equal weights, then let $\sigma\to0$. Responsibilities become 0 or 1 (each point goes wholly to its nearest mean), and the M step becomes "move each mean to the average of its points". That is exactly Lloyd's algorithm. So k-means is a GMM with round, equal, infinitely tight clusters and hard assignments.

## 3. From scratch in 1D

Two groups: 300 points around 0 with spread 1 and 200 points around 4 with spread 1.5. We never use the group labels in the fit.

In [ ]:
x1 = np.r_[rng.normal(0, 1.0, 300), rng.normal(4, 1.5, 200)]


def em_1d(x: np.ndarray, mu: np.ndarray, sd: np.ndarray, pi: np.ndarray, n_iter: int = 40):
    """EM for a 1D Gaussian mixture. Returns final parameters and the log-likelihood per step."""
    ll = []
    for _ in range(n_iter):
        dens = pi * norm.pdf(x[:, None], mu, sd)  # (n, K)
        ll.append(np.log(dens.sum(axis=1)).sum())
        r = dens / dens.sum(axis=1, keepdims=True)  # E step
        nk = r.sum(axis=0)  # M step
        pi = nk / len(x)
        mu = (r * x[:, None]).sum(axis=0) / nk
        sd = np.sqrt((r * (x[:, None] - mu) ** 2).sum(axis=0) / nk)
    return mu, sd, pi, np.array(ll)


mu, sd, pi, ll = em_1d(
    x1, mu=np.array([-1.0, 1.0]), sd=np.array([1.0, 1.0]), pi=np.array([0.5, 0.5])
)
print("means:", mu, " sds:", sd, " weights:", pi)
print(
    "log-likelihood never decreases:",
    bool(np.all(np.diff(ll) >= -1e-9)),
    " first, last:",
    ll[[0, -1]].round(1),
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
grid = np.linspace(-4, 9, 300)
axes[0].hist(x1, bins=40, density=True, alpha=0.4)
axes[0].plot(grid, (pi * norm.pdf(grid[:, None], mu, sd)).sum(axis=1), "k", label="fitted mixture")
for j in range(2):
    axes[0].plot(grid, pi[j] * norm.pdf(grid, mu[j], sd[j]), "--")
axes[0].legend()
axes[1].plot(ll)
axes[1].set(xlabel="EM iteration", ylabel="log-likelihood")
plt.show()

The fit recovers the generating values (means 0 and 4, spreads 1 and 1.5, weights 0.6 and 0.4) from a poor start, and the log-likelihood rises and flattens as EM promises.

## 4. From scratch in 2D

The same loop with full covariance matrices. Make three clusters that are tilted and of different sizes, which k-means handles badly.

In [ ]:
def make_tilted(seed: int) -> tuple[np.ndarray, np.ndarray]:
    r = np.random.default_rng(seed)
    parts, labels = [], []
    specs = [
        ([0, 0], [[2.5, 1.8], [1.8, 1.7]], 300),
        ([6, 1], [[1.5, -1.0], [-1.0, 1.0]], 200),
        ([2, 6], [[0.3, 0], [0, 0.3]], 100),
    ]
    for j, (m, c, n) in enumerate(specs):
        parts.append(r.multivariate_normal(m, c, n))
        labels += [j] * n
    return np.vstack(parts), np.array(labels)


X, y = make_tilted(SEED)


def em_gmm(X: np.ndarray, K: int, n_iter: int = 100, seed: int = 0):
    """EM for a full-covariance GMM, started from k-means centres. Returns pi, mu, cov, log-likelihoods."""
    n, d = X.shape
    km = KMeans(K, n_init=3, random_state=seed).fit(X)
    mu = km.cluster_centers_.copy()
    cov = np.array([np.cov(X[km.labels_ == k].T) for k in range(K)])
    pi = np.bincount(km.labels_, minlength=K) / n
    ll = []
    for _ in range(n_iter):
        dens = np.column_stack(
            [pi[k] * multivariate_normal.pdf(X, mu[k], cov[k]) for k in range(K)]
        )
        ll.append(np.log(dens.sum(axis=1)).sum())
        r = dens / dens.sum(axis=1, keepdims=True)  # E step
        nk = r.sum(axis=0)  # M step
        pi, mu = nk / n, (r.T @ X) / nk[:, None]
        cov = np.array(
            [
                ((r[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / nk[k] + 1e-6 * np.eye(d)
                for k in range(K)
            ]
        )
        if len(ll) > 1 and abs(ll[-1] - ll[-2]) < 1e-8:
            break
    return pi, mu, cov, np.array(ll), r


pi2, mu2, cov2, ll2, resp = em_gmm(X, 3)
ref = GaussianMixture(
    3,
    covariance_type="full",
    init_params="kmeans",
    n_init=1,
    random_state=SEED,
    tol=1e-8,
    max_iter=500,
).fit(X)
order = np.argsort(mu2[:, 0])
ref_order = np.argsort(ref.means_[:, 0])
print("iterations:", len(ll2))
print("means, mine:\n", mu2[order], "\nmeans, sklearn:\n", ref.means_[ref_order])
print("weights, mine vs sklearn:", pi2[order], ref.weights_[ref_order])
print("total log-likelihood, mine vs sklearn:", round(ll2[-1], 2), round(ref.score(X) * len(X), 2))

The two implementations land on the same means, weights and likelihood. Now compare the clustering with k-means and look at the soft assignments.

In [ ]:
def draw_ellipses(ax, means, covs, n_sd: float = 2.0) -> None:
    for m, c in zip(means, covs):
        vals, vecs = np.linalg.eigh(c)
        angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        ax.add_patch(
            Ellipse(
                m,
                2 * n_sd * np.sqrt(vals[1]),
                2 * n_sd * np.sqrt(vals[0]),
                angle=angle,
                fill=False,
                lw=2,
            )
        )


km = KMeans(3, n_init=10, random_state=SEED).fit(X)
gm_labels = resp.argmax(axis=1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
axes[0].scatter(*X.T, c=km.labels_, cmap="tab10", vmin=0, vmax=9, s=14)
axes[0].set_title(f"k-means: ARI {adjusted_rand_score(y, km.labels_):.2f}")
axes[1].scatter(*X.T, c=gm_labels, cmap="tab10", vmin=0, vmax=9, s=14)
draw_ellipses(axes[1], mu2, cov2)
axes[1].set_title(f"GMM, hard label: ARI {adjusted_rand_score(y, gm_labels):.2f}")
uncertain = resp.max(axis=1)
sc = axes[2].scatter(*X.T, c=uncertain, cmap="viridis", s=14)
axes[2].set_title("GMM confidence (max responsibility)")
fig.colorbar(sc, ax=axes[2])
plt.show()
print("points with max responsibility below 0.9:", int((uncertain < 0.9).sum()), "of", len(X))

k-means draws straight borders halfway between centres, so it hands part of the large tilted cluster to its neighbour. The GMM fits a tilted ellipse to each group and agrees much better with the truth (ARI 0.97 against 0.85). The right panel shows where it is unsure: points in the overlap between the two bigger clusters. k-means cannot say that.

## 5. Library version and covariance types

`GaussianMixture` has a `covariance_type` option that controls how flexible each Gaussian is.

| `covariance_type` | Shape per component | Parameters per component (2D) |
|---|---|---|
| `spherical` | circle, one variance | 1 |
| `diag` | axis-aligned ellipse | 2 |
| `tied` | same ellipse shape for every component | 3 shared |
| `full` | any tilted ellipse | 3 |

More flexibility fits more shapes but needs more data to estimate. `spherical` with equal weights is close to k-means.

In [ ]:
for ctype in ("spherical", "diag", "tied", "full"):
    g = GaussianMixture(3, covariance_type=ctype, n_init=5, random_state=SEED).fit(X)
    print(f"{ctype:>9}: ARI {adjusted_rand_score(y, g.predict(X)):.3f}  BIC {g.bic(X):.0f}")

## 6. Tuning: how many components?

Likelihood always improves as you add components, so you cannot pick the maximum, just as you cannot pick the lowest k-means inertia. The **Bayesian information criterion** adds a penalty:

$$\text{BIC}=p\ln n-2\ln\hat L$$

where $p$ is the number of free parameters and $\hat L$ the maximised likelihood. Lower is better. The first term charges for complexity, and the charge grows with $n$, so extra components must earn their keep. (The Akaike criterion, `aic`, uses $2p$ and penalises less.) Because a GMM has a likelihood, BIC is available. k-means has no likelihood, which is why it relies on the elbow and silhouette.

In [ ]:
ks = range(1, 9)
bic = [GaussianMixture(k, n_init=5, random_state=SEED).fit(X).bic(X) for k in ks]
aic = [GaussianMixture(k, n_init=5, random_state=SEED).fit(X).aic(X) for k in ks]
plt.plot(list(ks), bic, marker="o", label="BIC")
plt.plot(list(ks), aic, marker="s", label="AIC")
plt.xlabel("components")
plt.ylabel("score (lower is better)")
plt.legend()
plt.show()
print("best k by BIC:", list(ks)[int(np.argmin(bic))], " by AIC:", list(ks)[int(np.argmin(aic))])

BIC picks 3 components, matching how the data were made. After that the curve rises, because the extra components add parameters without improving the fit enough.

Other settings. `n_init` runs several starts and keeps the best likelihood, as in k-means. `reg_covar` adds a small value to each covariance diagonal for stability (see below). `init_params="kmeans"` (the default) starts from k-means centres.

## 7. What breaks

**Not Gaussian.** The model assumes each cluster is an ellipse. Moons are not.

In [ ]:
Xm_df, ym = make_moons_data(400, noise=0.08, seed=SEED)
Xm = Xm_df.to_numpy()
for k in (2, 6):
    g = GaussianMixture(k, n_init=5, random_state=SEED).fit(Xm)
    print(
        f"moons, {k} components: ARI {adjusted_rand_score(ym, g.predict(Xm)):.2f}  BIC {g.bic(Xm):.0f}"
    )
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, k in zip(axes, (2, 6)):
    g = GaussianMixture(k, n_init=5, random_state=SEED).fit(Xm)
    ax.scatter(*Xm.T, c=g.predict(Xm), cmap="tab10", vmin=0, vmax=9, s=14)
    draw_ellipses(ax, g.means_, g.covariances_)
    ax.set_title(f"{k} components")
plt.show()

With two components the ellipses cut across the moons (ARI well below 1). BIC prefers more components, because several ellipses can approximate a curve, but those components are pieces of a moon, not the moons. A good BIC says the density is modelled well, not that the components are meaningful groups.

**Singular covariance.** A component can collapse onto a single point (or a few identical ones). Its covariance shrinks to zero, its density goes to infinity and the likelihood blows up. This is the main way MLE for a GMM goes wrong. Here is a component pinned to one point.

In [ ]:
xs = np.r_[rng.normal(0, 1, 50), [3.0] * 4][:, None]  # four identical points
for reg in (0.0, 1e-6, 1e-2):
    g = GaussianMixture(2, reg_covar=reg, means_init=[[0.0], [3.0]], random_state=0).fit(xs)
    print(
        f"reg_covar={reg}: smallest variance {g.covariances_.min():.1e}, weights {g.weights_}, avg log-lik {g.score(xs):.2f}"
    )

With no floor the second component sits on the four identical points with a variance of essentially zero and a weight of 7%. Its density there is astronomically high, so the average log-likelihood (0.81) looks better than the regularised fits ($-1.19$ and $-1.53$), although the model is worse. `reg_covar` (default $10^{-6}$) puts a floor under each variance and keeps the fit finite. If a fitted component has a tiny weight and a tiny variance, treat it as a symptom, not a finding.

**Local optima.** Different starts can give different answers, so use `n_init` above 1.

**Dimension.** A full covariance has $d(d+1)/2$ parameters per component. With 50 features that is 1,275 per component, so a few hundred rows cannot support it. Use `diag`, or reduce dimensions first (notebook 04).

**When does a GMM beat k-means?** When clusters are elliptical or tilted, have different spreads or sizes, overlap, or when you want to know how sure the model is. When clusters are round and well separated the two give the same answer and k-means is faster. When clusters are not blob-shaped at all, neither is right and notebook 02 applies.

Next, the most widely used way to reduce dimensions: PCA.